In [ ]:
from auditwheel.architecture import Architecture
from auditwheel.libc import Libc
from auditwheel.policy import WheelPolicies

# Example 1: manylinux platform policies
print("Example 1: manylinux platform policies")

try:
    # Pass the libc and the architecture explicitly, so the result does not depend on this machine
    policies = WheelPolicies(libc=Libc.GLIBC, arch=Architecture.x86_64)
    print("✓ glibc policies for x86_64 loaded")

    # A legacy name and its PEP 600 name resolve to the same policy
    p2014 = policies.get_policy_by_name("manylinux2014_x86_64")
    assert p2014.name == "manylinux_2_17_x86_64", p2014.name
    assert "manylinux2014_x86_64" in p2014.aliases, p2014.aliases
    assert policies.get_policy_by_name("manylinux_2_17_x86_64").name == p2014.name
    print(f"✓ manylinux2014_x86_64 is {p2014.name}")

    # manylinux_2_17 allows glibc symbol versions up to 2.17 and nothing newer
    glibc = p2014.symbol_versions["GLIBC"]
    assert "2.17" in glibc and "2.18" not in glibc, sorted(glibc)
    assert "libc.so.6" in p2014.whitelist
    print(f"✓ {p2014.name} accepts {len(glibc)} GLIBC symbol versions, the newest being 2.17")

    p1 = policies.get_policy_by_name("manylinux1_x86_64")
    p228 = policies.get_policy_by_name("manylinux_2_28_x86_64")
    assert p1.name == "manylinux_2_5_x86_64", p1.name
    assert "2.28" in p228.symbol_versions["GLIBC"]
    # An older glibc baseline runs on more systems, so it has a higher priority
    assert policies.linux.name == "linux_x86_64" and policies.linux.priority == 0
    assert p1.priority > p2014.priority > p228.priority > policies.linux.priority
    print(f"✓ Priority order: {p1.name} > {p2014.name} > {p228.name} > {policies.linux.name}")

    # The same policy names exist for other architectures
    arm = WheelPolicies(libc=Libc.GLIBC, arch=Architecture.aarch64)
    assert arm.get_policy_by_name("manylinux2014_aarch64").name == "manylinux_2_17_aarch64"
    print("✓ manylinux2014_aarch64 is manylinux_2_17_aarch64")

    print("\nmanylinux tags and the distributions they were built on:")
    manylinux_policies = [
        "manylinux1_x86_64 = manylinux_2_5_x86_64 (CentOS 5)",
        "manylinux2010_x86_64 = manylinux_2_12_x86_64 (CentOS 6)",
        "manylinux2014_x86_64 = manylinux_2_17_x86_64 (CentOS 7)",
        "manylinux_2_28_x86_64 (AlmaLinux 8, glibc 2.28)"
    ]
    for policy_info in manylinux_policies:
        print(f"  • {policy_info}")

except Exception as e:
    print(f"Error in platform analysis: {e}")
    raise AssertionError(
        f"auditwheel example (cell 0) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 completed\n")

In [ ]:
## Example 2: Wheel File Analysis

from auditwheel.architecture import Architecture
from auditwheel.libc import Libc
from auditwheel.wheeltools import (
    NonPlatformWheelError,
    WheelToolsError,
    get_wheel_architecture,
    get_wheel_libc,
)
from packaging.utils import parse_wheel_filename

# Example 2: Wheel filename analysis
print("Example 2: Wheel File Analysis")

try:
    # auditwheel reads the architecture and the C library from the platform tags of a wheel's file name
    cases = {
        "numpy-2.0.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl": (Architecture.x86_64, Libc.GLIBC),
        "scipy-1.7.0-cp38-cp38-manylinux1_x86_64.whl": (Architecture.x86_64, Libc.GLIBC),
        "package-1.0-cp312-cp312-musllinux_1_2_aarch64.whl": (Architecture.aarch64, Libc.MUSL),
    }
    print("Wheel filename analysis:")
    for wheel_name, (arch, libc) in cases.items():
        found = (get_wheel_architecture(wheel_name), get_wheel_libc(wheel_name))
        print(f"  {wheel_name}: {found[0].value}, {found[1].value}")
        assert found == (arch, libc), (wheel_name, found)

    # A pure-Python wheel has no platform, so it has no architecture and no libc
    pure = "package-1.0.0-py3-none-any.whl"
    try:
        get_wheel_architecture(pure)
    except NonPlatformWheelError:
        print(f"  {pure}: not a platform wheel")
    else:
        raise AssertionError(f"{pure} was given an architecture")
    try:
        get_wheel_libc(pure)
    except WheelToolsError:
        pass
    else:
        raise AssertionError(f"{pure} was given a libc")

    # The tags themselves come from packaging, which auditwheel builds on
    name, version, build, tags = parse_wheel_filename(next(iter(cases)))
    assert (name, str(version), build) == ("numpy", "2.0.0", ()), (name, version, build)
    assert {tag.platform for tag in tags} == {"manylinux_2_17_x86_64", "manylinux2014_x86_64"}, tags
    print(f"✓ numpy 2.0.0 wheel carries {len(tags)} tags: {sorted(str(tag) for tag in tags)}")

    print("✓ Wheel analysis checks passed")

except Exception as e:
    print(f"Error in wheel analysis: {e}")
    raise AssertionError(
        f"auditwheel example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 2 completed\n")

In [ ]:
import sys
from pathlib import Path
from auditwheel.lddtree import ldd

# Example 3: Dependency tree of an ELF file (the running Python interpreter)
print("Example 3: Wheel Dependency Analysis")

try:
    # auditwheel builds the same tree for every shared library it finds inside a wheel
    tree = ldd(Path(sys.executable))
    print(f"Interpreter: {sys.executable}")
    print(f"  ELF interpreter (dynamic loader): {tree.interpreter}")
    print(f"  Libraries it needs directly: {list(tree.needed)}")

    # Every direct dependency is resolved to a file on this system: the interpreter is running, so they exist
    missing = [name for name in tree.needed if name not in tree.libraries or tree.libraries[name].realpath is None]
    assert not missing, f"unresolved dependencies: {missing}"
    for name in tree.needed:
        library = tree.libraries[name]
        assert Path(library.realpath).is_file(), (name, library.realpath)
        print(f"  {name} -> {library.realpath}")

    print("✓ Dependency tree resolved")

except Exception as e:
    print(f"Error in dependency analysis: {e}")
    raise AssertionError(
        f"auditwheel example (cell 2) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 3 completed\n")

In [ ]:
import base64
import hashlib
import os
import subprocess
import sys
import tempfile
import zipfile

# Example 4: The auditwheel command line on a pure-Python wheel
print("Example 4: Wheel Repair Operations")

try:
    work = tempfile.mkdtemp(prefix="auditwheel_nb_")
    wheel_path = os.path.join(work, "demo_pkg-1.0-py3-none-any.whl")

    # A minimal pure-Python wheel: one module, METADATA, WHEEL and RECORD
    files = {
        "demo_pkg/__init__.py": b"VALUE = 42\n",
        "demo_pkg-1.0.dist-info/METADATA": b"Metadata-Version: 2.1\nName: demo-pkg\nVersion: 1.0\n",
        "demo_pkg-1.0.dist-info/WHEEL": b"Wheel-Version: 1.0\nGenerator: notebook\nRoot-Is-Purelib: true\nTag: py3-none-any\n",
    }

    def record_hash(data):
        digest = base64.urlsafe_b64encode(hashlib.sha256(data).digest()).rstrip(b"=").decode()
        return f"sha256={digest},{len(data)}"

    record = "".join(f"{name},{record_hash(data)}\n" for name, data in files.items())
    record += "demo_pkg-1.0.dist-info/RECORD,,\n"
    with zipfile.ZipFile(wheel_path, "w") as wheel:
        for name, data in files.items():
            wheel.writestr(name, data)
        wheel.writestr("demo_pkg-1.0.dist-info/RECORD", record)
    print(f"✓ Built {os.path.basename(wheel_path)}")

    def auditwheel(*args):
        return subprocess.run([sys.executable, "-m", "auditwheel", *args], capture_output=True, text=True, timeout=120)

    # `auditwheel show` finds no compiled code, so there is nothing to audit
    shown = auditwheel("show", wheel_path)
    print(f"auditwheel show -> exit code {shown.returncode}")
    assert shown.returncode != 0, shown
    assert "platform wheel" in shown.stdout + shown.stderr, shown

    # `auditwheel repair` refuses the same wheel and writes nothing to the output directory
    wheelhouse = os.path.join(work, "wheelhouse")
    repaired = auditwheel("repair", "--wheel-dir", wheelhouse, wheel_path)
    print(f"auditwheel repair -> exit code {repaired.returncode}")
    assert repaired.returncode != 0, repaired
    assert "platform wheel" in repaired.stdout + repaired.stderr, repaired
    assert not os.path.isdir(wheelhouse) or not os.listdir(wheelhouse), os.listdir(wheelhouse)
    print("✓ Pure-Python wheel reported as not a platform wheel; nothing to repair")

    # What repair does for a wheel with compiled extensions
    print("\nWheel repair process stages:")
    repair_stages = [
        "1. Analyze wheel dependencies with lddtree",
        "2. Identify external libraries not in manylinux", 
        "3. Copy required shared libraries into wheel",
        "4. Update RPATH/RUNPATH entries for bundled libs",
        "5. Modify wheel metadata and platform tags",
        "6. Generate manylinux-compatible wheel output"
    ]
    for stage in repair_stages:
        print(f"  {stage}")

except Exception as e:
    print(f"Error in repair operations: {e}")
    raise AssertionError(
        f"auditwheel example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 4 completed\n")

In [ ]:
import auditwheel
import auditwheel.policy as policy

# Example 5: Auditwheel Package Capabilities and Linux Wheel Distribution
print("Example 5: Auditwheel Package Capabilities")

try:
    print("auditwheel - Tool for auditing and repairing Python wheels on Linux")
    
    # Show package version and components
    # auditwheel exports no `__version__` attribute; the installed distribution's metadata carries it
    from importlib.metadata import version as distribution_version
    version = distribution_version("auditwheel")
    print(f"auditwheel version: {version}")
    assert version[:1].isdigit(), f"unexpected auditwheel version string {version!r}"
    
    # Key auditwheel capabilities
    print("\nKey auditwheel capabilities:")
    capabilities = [
        "🔍 Audit wheels for manylinux compliance",
        "🔧 Repair non-compliant wheels automatically", 
        "📦 Bundle external shared libraries into wheels",
        "🏷️ Update platform tags for compatibility",
        "📊 Analyze wheel dependencies and symbols",
        "🎯 Target specific manylinux policies",
        "🔗 Manage RPATH and library linking",
        "✅ Validate wheel distribution readiness"
    ]
    
    for capability in capabilities:
        print(f"  {capability}")
    
    # Manylinux policies supported
    print("\nManylinux policies and targets:")
    manylinux_info = [
        "manylinux1_x86_64 - CentOS 5 compatibility (oldest)",
        "manylinux2010_x86_64 - CentOS 6 compatibility",
        "manylinux2014_x86_64 - CentOS 7 compatibility", 
        "manylinux_2_17_x86_64 - same as manylinux2014 (CentOS 7)",
        "manylinux_2_24_x86_64 - Debian 9 compatibility",
        "manylinux_2_28_x86_64 - Debian 10 compatibility"
    ]
    
    for policy_info in manylinux_info:
        print(f"  • {policy_info}")
    
    # Common use cases
    print("\nCommon auditwheel use cases:")
    use_cases = [
        "🏭 CI/CD pipeline wheel compliance checking",
        "📦 PyPI package preparation and validation",
        "🔄 Cross-platform wheel compatibility ensuring", 
        "🛠️ Native extension distribution optimization",
        "📊 Dependency analysis for containerized apps",
        "🎯 Library bundling for offline deployments",
        "✨ Automated wheel repair in build systems",
        "🔍 Debugging wheel import and runtime issues"
    ]
    
    for use_case in use_cases:
        print(f"  {use_case}")
    
    # Integration patterns
    print("\nIntegration with build tools:")
    integrations = [
        "• setuptools and setup.py builds",
        "• pip wheel building workflows",
        "• cibuildwheel for cross-platform builds", 
        "• GitHub Actions and CI pipelines",
        "• Docker-based build environments",
        "• conda-build package creation"
    ]
    
    for integration in integrations:
        print(f"  {integration}")
    
    print("\nauditwheel ensures Linux wheels work across distributions")
    print("Essential for professional Python package distribution")
    print("auditwheel example 5 completed successfully")
    
except Exception as e:
    print(f"Error in capabilities example: {e}")
    raise AssertionError(
        f"auditwheel example (cell 4) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")